# noise_reachability step 10: can constant-noise successes be learned into the weights? (Kaggle, 2 x T4)

Step 10 of `noise_reachability/README.md` (design fixed 2026-11-03). The base `lerobot/smolvla_libero` at `a24f1f3804`,
`n_action_steps=10`, tasks 0, 6, 7: replays of earlier constant-noise successes are recorded, LoRA adapters are trained
on them (arm N) and on the base's own successes (arm A, task 0), and every policy is evaluated with sampled noise.

**Run with Save & Run All (Accelerator: GPU T4 x2, Internet on).** Nothing needs editing. The wrapper check and the
harness run first; a failure of either stops the run. Each session stops starting work after 9 h and then writes the zip.

**This run needs about four sessions.** When step 15 says work remains, add this version's output as input (**Add Input
> Your Work**), and only that version, and Save & Run All again: finished episodes and adapters are kept. Step 6 must
then print `RESUMING`.

Download from the Output tab: `step10_results.zip` (results without recordings or adapter weights, which stay in the
version's output for the next session).

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "f4369f3e8514232fef3d047a1ede6c1b6287267d"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 10; nothing here is meant to be edited. The analysis refuses anything else.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASKS = [0, 6, 7]
N_ACTION_STEPS = 10
COLLECT_PARTS = ["10-19", "20-29", "30-39", "40-49"]   # arm A's data, as step 4: task 0, one process each
COLLECT_SEEDS = "0-2"
VAL_INIT_STATES = "45-49"          # arm A's validation episodes, as step 4
TRAIN_SEEDS = [0, 1, 2]
TRAIN_STEPS, BATCH_SIZE = 2000, 16 # step 4's settings, unchanged
EVAL_STATES = {0: "0-9,11,14,27", 6: "0,8,12,14,20", 7: "4,33,38,47"}
EVAL_SEED_PARTS = {0: ["16-19", "20-23", "24-27", "28-31"],
                   6: ["16-19", "20-23", "24-27", "28-31", "32-35", "36-39", "40-43", "44-47"],
                   7: ["16-19", "20-23", "24-27", "28-31", "32-35", "36-39", "40-43", "44-47"]}
OUT_NAME = "step10"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"
STOP_AT_H = 9.0                    # no new work after this many hours of the session; running jobs are stopped
TRAIN_NEEDS_H = 3.5                # before the timing check: no training run starts with less time left
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
# Adapters in training order: arm A first (its data is cheap), then arm N one task after another per seed.
ADAPTERS = [f"A{s}" for s in TRAIN_SEEDS] + [f"N{t}_{s}" for s in TRAIN_SEEDS for t in TASKS]
# Policies per task under analyze_step10.py's names (N<s> is adapter N<t>_<s> on task t).
POLICIES = {t: ["base"] + [f"N{s}" for s in TRAIN_SEEDS] + ([f"A{s}" for s in TRAIN_SEEDS] if t == 0 else [])
            for t in TASKS}


def adapter_of(t, policy):
    return policy if policy.startswith("A") else f"N{t}_{policy[1:]}"
print(f"step 10 at {REPO_REF}: tasks {TASKS}, n_action_steps {N_ACTION_STEPS}, adapters {ADAPTERS}, "
      f"at most {90 + 122 + 120 + 2608} episodes (90 harness, 122 replays, 120 collection, 2608 evaluation), "
      f"stop after {STOP_AT_H} h, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
assert JOBS >= 4, f"RAM for {JOBS} SmolVLA processes; the notebook runs 4 at once"
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

The package versions of the local `lerobot_env` (as the earlier Kaggle notebooks). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("base checkpoint", POLICY_PATH)

## 6. Resume from an earlier version's output

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
previous = [p for p in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True))
            if os.path.isdir(p) and os.path.exists(f"{p}/repo_commit.txt")]
assert len(previous) <= 1, f"attach only the latest version's output, found {previous}"
for prev in previous:
    print("RESUMING from", prev)
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
if not previous:
    print("FRESH START: no earlier output attached. If this is the second session, stop and add it as input.", flush=True)
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the run started at commit {started_with}, this version cloned {head}: set REPO_REF in "
                       f"step 1 to {started_with}")
with open(first, "a") as f:
    f.write(head + "\n")
for d in ("logs", "eval"):
    os.makedirs(f"{OUT_ROOT}/{d}", exist_ok=True)

## 7. Helpers: parallel jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 16, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)






SEARCH = f"cd {REPO} && {PY} -u noise_reachability/deep_noise_search.py --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}"
ANALYZE = f"cd {REPO} && {PY} -S noise_reachability/analyze_step10.py --root {OUT_ROOT}"
MAIN = f"{REPO}/self_filtered_bc/self_filtered_bc.py"


def task_dir(task):
    return f"{OUT_ROOT}/task_{task}"


def sfbc(task):
    return f"cd {REPO} && {PY} -u {MAIN} --task_id {task} --n_action_steps {N_ACTION_STEPS} --policy_path {POLICY_PATH}"


def run_stage(name, jobs, total, max_parallel=4):
    """True when the stage has no work left: a job that failed stops the run, the deadline leaves work for a next version."""
    print(f"{name}: {len(jobs)} of {total} jobs to run; {hours_left():.1f} h left", flush=True)
    if not jobs:
        return True
    if hours_left() <= 0:
        return False
    codes = run_jobs(jobs, max_parallel=max_parallel)
    if failed(codes):
        fail(f"{name} failed: {failed(codes)}")
        return False
    return all(c == 0 for c in codes.values()) and len(codes) == len(jobs)


def run_gate(which):
    if sh(f"{ANALYZE} --gate {which}", check=False, env=RUN_ENV).returncode != 0:
        fail(f"the {which} gate failed: investigate before spending more")
        return False
    return True


def adapter_done(name):
    return Path(OUT_ROOT, "runs", name, "adapter", "run_config.json").exists()

## 8. Wrapper check on each task's real model (a few minutes); stops the run if it fails

In [ ]:
CHECK_DONE = True
for t in TASKS:
    r = sh(f"cd {REPO} && {PY} -u noise_reachability/check_step5a.py --task_id {t} --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}",
           check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"check_step5a failed on task {t}")
        CHECK_DONE = False

## 9. Harness: the base on seeds 0-2, per task; stops the run if it fails

In [ ]:
jobs = {}
for t in TASKS:
    for s in (0, 1, 2):
        out = f"{task_dir(t)}/harness/seed{s}"
        if len(read_csv(f"{out}/per_episode_base.csv")) < 10:
            jobs[f"harness_task{t}_seed{s}"] = (
                f"cd {REPO} && {PY} -u noise_reachability/eval_noise_selection.py --task_id {t} --policies base "
                f"--policy_path {POLICY_PATH} --seeds {s} --init_states 0-9 --n_action_steps {N_ACTION_STEPS} --output_dir {out}",
                len(jobs) % 2)
HARNESS_DONE = CHECK_DONE and run_stage("harness", jobs, 3 * len(TASKS)) and run_gate("harness")

## 10. Arm N's data: replay and record every constant-noise success on a resistant state

The list comes from `analyze_step10.py --replay_jobs` (from the committed result files); recordings go to
`task_<t>/replay_data/episodes/`.

In [ ]:
REPLAYS_DONE = False
if HARNESS_DONE:
    r = sh(f"cd {REPO} && {PY} -S noise_reachability/analyze_step10.py --replay_jobs", check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail("the replay list failed")
    else:
        plan = [line.split() for line in r.stdout.strip().splitlines()]
        jobs = {}
        for t, f, s, ids, js in plan:
            out = f"{task_dir(t)}/replay"
            n = len((ids if js == "-" else js).split(","))
            csv_name = f"per_episode_{f}_state{s}.csv" if js == "-" else f"confirm_{f}_n{ids}_state{s}.csv"
            if len(read_csv(f"{out}/{csv_name}")) < n:
                jitter = "" if js == "-" else f" --jitter_seeds {js}"
                jobs[f"replay_task{t}_{f}_state{s}_{ids if js != '-' else 'plain'}"] = (
                    f"{SEARCH} --task_id {t} --family {f} --init_state {s} --noise_ids {ids}{jitter} --output_dir {out} "
                    f"--record_dir {task_dir(t)}/replay_data", len(jobs) % 2)
        print(f"{len(plan)} replay jobs, {sum(len((i if j == '-' else j).split(',')) for _, _, _, i, j in plan)} episodes",
              flush=True)
        REPLAYS_DONE = run_stage("replays", jobs, len(plan))
        for t in TASKS:
            rows = [r for p in glob.glob(f"{task_dir(t)}/replay/*.csv") for r in read_csv(p)]
            print(f"task {t}: {sum(r['success'] == '1' for r in rows)} of {len(rows)} replays succeeded; "
                  f"{len([f for f in glob.glob(f'{task_dir(t)}/replay_data/episodes/*.npz') if not f.endswith('.tmp.npz')])} "
                  f"recordings", flush=True)

## 11. Arm A's data: the base on task 0, init states 10-49 x seeds 0-2 (as step 4)

In [ ]:
DATA_A = f"{task_dir(0)}/collect_data"
need = 40 * n_range(COLLECT_SEEDS)
COLLECTED = len(read_csv(f"{DATA_A}/collection.csv")) == need
if HARNESS_DONE and not COLLECTED:
    jobs = {}
    for k, part in enumerate(COLLECT_PARTS):
        if len(read_csv(f"{task_dir(0)}/collect/part{k}/collection.csv")) < 10 * n_range(COLLECT_SEEDS):
            jobs[f"collect_{k}"] = (f"{sfbc(0)} --stage collect --data_dir {task_dir(0)}/collect/part{k} --init_states {part} "
                                    f"--collect_seeds {COLLECT_SEEDS} --max_episodes_per_process 15", k % 2)
    if run_stage("collection", jobs, len(COLLECT_PARTS)):
        rows = [r for k in range(len(COLLECT_PARTS)) for r in read_csv(f"{task_dir(0)}/collect/part{k}/collection.csv")]
        if len(rows) == need:
            os.makedirs(f"{DATA_A}/episodes", exist_ok=True)
            for f in glob.glob(f"{task_dir(0)}/collect/part*/episodes/*.npz"):
                if f.endswith(".tmp.npz"):     # a write cut off by a stopped process
                    continue
                shutil.move(f, f"{DATA_A}/episodes/{os.path.basename(f)}")
            write_csv(rows, f"{DATA_A}/collection.csv")
            COLLECTED = len([f for f in glob.glob(f"{DATA_A}/episodes/*.npz") if not f.endswith(".tmp.npz")]) == need
if COLLECTED:
    rows = read_csv(f"{DATA_A}/collection.csv")
    print(f"arm A's data: {sum(r['success'] == '1' for r in rows)} of {len(rows)} successful")

## 12. Training: arm A (task 0) and arm N (tasks 0, 6, 7), training seeds 0-2

Two adapters at a time, one per T4. A run starts only if the session can finish it (1.25 x the measured time plus 10
minutes); an unfinished run restarts from scratch in the next session.

In [ ]:
TIMING = Path(OUT_ROOT, "train_timing.json")
if (REPLAYS_DONE or COLLECTED) and not TIMING.exists() and hours_left() > 0.5:
    data = DATA_A if COLLECTED else f"{task_dir(7)}/replay_data"
    task = 0 if COLLECTED else 7
    r = sh(f"{sfbc(task)} --stage train --seed 0 --selection success --data_dir {data} --train_steps 20 "
           f"--batch_size {BATCH_SIZE} --log_every 5 --val_every 1000 --output_dir /tmp/step10_train_timing",
           check=False, env=dict(RUN_ENV, CUDA_VISIBLE_DEVICES="0"))
    found = re.findall(r"sec_per_step=([0-9.]+)", r.stdout)
    if r.returncode != 0 or not found:
        fail("the training timing run failed")
    else:
        TIMING.write_text(json.dumps({"sec_per_step": float(found[-1]), "date": time.strftime("%Y-%m-%d %H:%M")}) + "\n")
if TIMING.exists():
    TRAIN_NEEDS_H = 1.25 * TRAIN_STEPS * json.loads(TIMING.read_text())["sec_per_step"] / 3600 + 10 / 60
print(f"a training run starts only with more than {TRAIN_NEEDS_H:.2f} h left")


def train_cmd(name):
    common = f"--stage train --selection success --train_steps {TRAIN_STEPS} --batch_size {BATCH_SIZE}"
    if name.startswith("A"):
        return (f"{sfbc(0)} {common} --seed {name[1:]} --data_dir {DATA_A} --val_init_states {VAL_INIT_STATES} "
                f"--output_dir {OUT_ROOT}/runs/{name}")
    t, s = name[1:].split("_")
    return f"{sfbc(t)} {common} --seed {s} --data_dir {task_dir(t)}/replay_data --output_dir {OUT_ROOT}/runs/{name}"


ready = [n for n in ADAPTERS if not adapter_done(n) and (COLLECTED if n.startswith("A") else REPLAYS_DONE)]
print("adapters to train now:", ready or "none", flush=True)
while ready and hours_left() > TRAIN_NEEDS_H and not FAILURE:
    batch, ready = ready[:2], ready[2:]
    for n in batch:
        shutil.rmtree(f"{OUT_ROOT}/runs/{n}", ignore_errors=True)   # an unfinished run restarts from scratch
    codes = run_jobs({f"train_{n}": (train_cmd(n), k) for k, n in enumerate(batch)}, max_parallel=2)
    if failed(codes):
        fail(f"training failed: {failed(codes)}")
    for n in batch:
        lines = open(f"{OUT_ROOT}/logs/train_{n}.log").read().splitlines()
        print(n, [l for l in lines if "training episodes" in l or "training frames" in l or "peak GPU" in l])
DONE_ADAPTERS = [n for n in ADAPTERS if adapter_done(n)]
print("adapters done:", DONE_ADAPTERS)

## 13. Evaluation with sampled noise: every policy whose adapter is ready

In [ ]:
def eval_out(t, policy, j):
    return f"{task_dir(t)}/eval/{policy}/part{j}"


def eval_cmd(t, policy, j):
    common = (f"{sfbc(t)} --stage eval --eval_seeds {EVAL_SEED_PARTS[t][j]} --eval_init_states {EVAL_STATES[t]} "
              f"--eval_output_dir {eval_out(t, policy, j)} --max_episodes_per_process 10")
    if policy == "base":
        return f"{common} --eval_policies base"
    return f"{common} --eval_policies lora --output_dir {OUT_ROOT}/runs/{adapter_of(t, policy)}"


def n_states(spec):
    return sum(n_range(p) for p in spec.split(","))


def part_size(t, j):
    return n_range(EVAL_SEED_PARTS[t][j]) * n_states(EVAL_STATES[t])


def eval_rows(t, policy):
    name = "base" if policy == "base" else "lora"
    return [r for j in range(len(EVAL_SEED_PARTS[t])) for r in read_csv(f"{eval_out(t, policy, j)}/per_episode_{name}.csv")]


jobs, k = {}, 0
if HARNESS_DONE and not FAILURE:
    for t in TASKS:
        for policy in POLICIES[t]:
            if policy != "base" and not adapter_done(adapter_of(t, policy)):
                continue
            name = "base" if policy == "base" else "lora"
            for j in range(len(EVAL_SEED_PARTS[t])):
                if len(read_csv(f"{eval_out(t, policy, j)}/per_episode_{name}.csv")) < part_size(t, j):
                    jobs[f"eval_task{t}_{policy}_part{j}"] = (eval_cmd(t, policy, j), k % 2)
                    k += 1
    run_stage("evaluation", jobs, len(jobs))
for t in TASKS:
    print(f"task {t}:", {p: len(eval_rows(t, p)) for p in POLICIES[t]},
          f"of {sum(part_size(t, j) for j in range(len(EVAL_SEED_PARTS[t])))} each")
ALL_DONE = (len(DONE_ADAPTERS) == len(ADAPTERS) and all(
    len(eval_rows(t, p)) == sum(part_size(t, j) for j in range(len(EVAL_SEED_PARTS[t]))) for t in TASKS for p in POLICIES[t]))

## 14. Analysis (when every episode is done)

In [ ]:
if ALL_DONE and not FAILURE:
    if sh(f"{ANALYZE} --out {OUT_ROOT}/step10_report.txt", check=False, env=RUN_ENV).returncode != 0:
        fail("the analysis failed: the episodes are in the zip")

## 15. Work left

In [ ]:
if not ALL_DONE and not FAILURE:
    print("WORK REMAINS: add this version's output as input and Save & Run All again")

## 16. Files for the repo

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step10_results.zip . -x '*.npz' '*.safetensors' '*.pt'")
sh(f"du -h /kaggle/working/step10_results.zip; du -sh {OUT_ROOT}")
print(f"session time used: {(time.time() - T_SESSION) / 3600:.1f} h")
if FAILURE:
    raise RuntimeError(FAILURE)